<a href="https://colab.research.google.com/github/Syed8855/FlyRank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

# One row = one (report_date, client_hash_id, content_hash_id) combination in fact_content_daily_performance
# — daily grain, one row per content item per client per calendar day.
# Time window: month=2026-03 (mid-panel). Verified below in Section 3.
# The sealed final month (2026-06, the _sample file) is never used for label/feature development — held out as test.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

# LABEL
* is_declining_label — derived downstream from trend in gsc_clicks/gsc_impressions
* over a rolling window; not a raw column in this table, constructed during rollup.

# FEATURES (knowable at decision moment — trailing/historical only)
* gsc_clicks, gsc_impressions, gsc_avg_position — core search performance signals
* ga4_sessions, ga4_engaged_sessions — engagement signals
* scroll_events — behavioral signal
* sessions_organic — traffic-source signal

# CONTEXT (identifiers/metadata, not features)
* client_hash_id, content_hash_id, report_date
* client_has_gsc, client_has_ga4, gsc_data_available,    ga4_data_available — availability flags used for filtering, not modeling

# EXCLUDED
* gsc_sum_position — raw numerator behind gsc_avg_position; redundant, avg_position already captures this
* Any future-window rollup columns (e.g. trend_direction, trend_pct) built later in the pipeline —
*  excluded because they are computed from the same outcome the label measures, i.e. they encode the answer directly.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [30]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

In [31]:
from datasets import load_dataset
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

### Claim: grain is one row per (report_date, client, content item)
Verifying no duplicate (date, client, content) combinations exist in month=2026-03.


### Claim: grain is one row per (report_date, client, content item)
Verifying no duplicate (date, client, content) combinations exist in month=2026-03.

In [32]:
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
    SELECT COUNT(*)
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

,count_star()
0,9841378


### Claim: slice covers exactly one calendar month, March 2026
Confirming row count and that dates span 2026-03-01 to 2026-03-31 with no bleed into other months.

In [33]:
con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 1
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [34]:

con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as row_count
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


In [35]:
con.sql(f"""
    SELECT COUNT(*) as total_rows, MIN(report_date) as start_date, MAX(report_date) as end_date
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


### Claim: not all rows have usable GSC data — availability must be filtered explicitly
Checking how many of the 9,841,378 total rows have gsc_data_available IS TRUE.

In [36]:
con.sql(f"""
    SELECT COUNT(*) as available_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


###Result:
grain confirmed (0 duplicate rows), slice is exactly March 2026 (9,841,378 rows),
and only ~37% (3,611,061) of rows have available GSC data — most content-client-day
combinations lack live Search Console sync on any given day.

In [37]:
features_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_trailing_month,
        SUM(gsc_impressions) AS impressions_trailing_month,
        AVG(gsc_avg_position) AS avg_position_trailing_month,
        SUM(ga4_sessions) AS sessions_trailing_month,
        SUM(scroll_events) AS scroll_events_trailing_month
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

features_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,clicks_trailing_month,impressions_trailing_month,avg_position_trailing_month,sessions_trailing_month,scroll_events_trailing_month
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,7.0,6523.0,7.209549,1.0,0.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,0.0,453.0,2.987198,0.0,0.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,6.0,5630.0,6.724039,3.0,0.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,13.0,4944.0,7.244844,2.0,0.0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,1.0,429.0,4.209227,2.0,1.0


- clicks_trailing_month — knowable at month-end because it's a sum of already-occurred daily clicks
- impressions_trailing_month — same: backward-looking sum, no future data
- avg_position_trailing_month — average of past daily positions, already observed
- sessions_trailing_month — GA4 sessions summed from the past, no future leakage
- scroll_events_trailing_month — behavioral signal, purely historical

### The leakage trap
Adding one label-derived column on purpose, to see the score jump artificially, then removing it.

In [38]:
import pandas as pd
from sklearn.linear_model import LogisticRegression

# Build a toy label: "declining" = below-median clicks this month (proxy, for demo purposes only)
features_df['is_declining_label'] = (features_df['clicks_trailing_month'] <= features_df['clicks_trailing_month'].median()).astype(int)

# Honest features (no leakage)
X_honest = features_df[['impressions_trailing_month', 'avg_position_trailing_month']].fillna(0)
y = features_df['is_declining_label']

model = LogisticRegression().fit(X_honest, y)
honest_score = model.score(X_honest, y)
print("Honest score:", honest_score)

# Now inject the trap: a column derived directly from the label itself
features_df['leaky_column'] = features_df['clicks_trailing_month']  # literally what defines the label

X_leaky = features_df[['impressions_trailing_month', 'avg_position_trailing_month', 'leaky_column']].fillna(0)
model_leaky = LogisticRegression().fit(X_leaky, y)
leaky_score = model_leaky.score(X_leaky, y)
print("Leaky score:", leaky_score)

Honest score: 0.8303477463816497
Leaky score: 1.0


In [39]:
# TRAP REMOVED — keeping only honest, pre-decision-moment features
X_final = features_df[['impressions_trailing_month', 'avg_position_trailing_month']].fillna(0)
model_final = LogisticRegression().fit(X_final, y)
print("Final honest score (leak removed):", model_final.score(X_final, y))

Final honest score (leak removed): 0.8303477463816497


### Leakage lesson
Adding `leaky_column` (a direct copy of `clicks_trailing_month`, the same signal the label is derived from)
pushed the score from 0.83 to a perfect 1.0 — not because the model learned anything real, but because it
was handed the answer disguised as a feature. This mirrors why `trend_direction`/`trend_pct` must stay
excluded from real modeling: any column computed from the same outcome the label measures will inflate
scores artificially and produce a model that looks great in evaluation but is useless in production,
since that future information isn't available at the actual decision moment.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

1. GSC coverage gap — only ~37% of daily rows (3,611,061 / 9,841,378) have gsc_data_available IS TRUE.
   This means the majority of content-client-day combinations lack live Search Console sync on any
   given day. Any model trained on GSC-derived features effectively learns only from the subset of
   content that has consistent tracking — likely biased toward larger/more established clients.

2. Unbalanced panel history — per the dataset card, per-client history depth differs
   (see dim_clients.gsc_data_start / ga4_data_start). A page with 6 months of history and a page
   onboarded last month look identical in a single-month slice, but their "decline" signal means
   different things — one has a stable baseline to decline from, the other doesn't.

3. Single-month snapshot risk — March 2026 alone can't separate genuine content decline from
   seasonal effects, one-off algorithm updates, or short-term client-side changes (e.g. a site
   migration) that happened to land in that window.

4. This data measures correlation, not causation — clicks/impressions/position moving together
   says nothing about *why* (algorithm change, competitor action, seasonality, content quality).
   Findings here are observed and directional, not causal claims about Google's ranking behavior.

5. GA4 availability is independent of GSC availability — sessions_trailing_month and
   scroll_events_trailing_month came back NaN for client-content pairs where gsc_data_available
   was TRUE but ga4_data_available was FALSE. Filtering on gsc_data_available alone does not
   guarantee GA4-derived features are populated; a real model needs to filter or impute each
   source's availability independently, not assume one flag covers both.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.